In [ ]:
from pyspark.sql.functions import lit, col

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_orders_payments_table", "olist_orders_payments")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("orders_payments_table", "orders_payments_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_payments_table_name = dbutils.widgets.get("raw_olist_orders_payments_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_payments_table_name = dbutils.widgets.get("orders_payments_table")

In [ ]:
raw_olist_orders_payments_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_payments_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{raw_olist_orders_payments_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{raw_olist_orders_payments_table_name} (
            orderId STRING,
            paymentSequential INT,
            paymentType STRING,
            paymentInstallments STRING,
            paymentValue TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_payments_silver_df = (
    raw_olist_orders_payments_df
    .where((col("order_id").rlike("^[0-9a-fA-F]{32}$")))
    .select(
        col("order_id").alias("orderId"),
        col("payment_sequential").alias("paymentSequential"),
        col("payment_type").alias("paymentType"),
        col("payment_installments").alias("paymentInstallments"),
        col("payment_value").alias("paymentValue")
    )
)

In [ ]:
orders_payments_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{orders_payments_table_name}")